<a href="https://colab.research.google.com/github/kessa8691-sudo/assignment-no-1-/blob/main/kidney_model_final_h5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "NOT available")

TensorFlow version: 2.20.0
GPU is available


In [2]:
import os

# Define where the datasets were downloaded
path1 = "/root/.cache/kagglehub/datasets/imtkaggleteam/kidney-stone-classification-and-object-detection/versions/1"
path2 = "/root/.cache/kagglehub/datasets/makiw4/kidney-dataset-stone-and-normal/versions/1"

print(f"Dataset 1 ready at: {path1}")
print(f"Dataset 2 ready at: {path2}")

Dataset 1 ready at: /root/.cache/kagglehub/datasets/imtkaggleteam/kidney-stone-classification-and-object-detection/versions/1
Dataset 2 ready at: /root/.cache/kagglehub/datasets/makiw4/kidney-dataset-stone-and-normal/versions/1


In [4]:
import os
import tensorflow as tf
import kagglehub

# 1. Download and get the EXACT path automatically
path1 = kagglehub.dataset_download("imtkaggleteam/kidney-stone-classification-and-object-detection")
path2 = kagglehub.dataset_download("makiw4/kidney-dataset-stone-and-normal")

# 2. Check what is inside path2 to make sure we point to the images
print("Files in path2:", os.listdir(path2))

# 3. Load the data using the automatic path
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    path2,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    path2,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

100%|██████████| 215M/215M [00:02<00:00, 101MB/s]

Extracting files...


100%|██████████| 216M/216M [00:01<00:00, 159MB/s]

Extracting files...


Files in path2: ['Kidney Ultrasound Images Stone and No Stone']
Found 9416 files belonging to 1 classes.
Using 7533 files for training.
Found 9416 files belonging to 1 classes.
Using 1883 files for validation.


In [5]:
# Update the path to go inside the sub-folder
new_path2 = os.path.join(path2, 'Kidney Ultrasound Images Stone and No Stone')

# Check what's inside now (should show 'Stone' and 'Normal')
print("Folders found:", os.listdir(new_path2))

# Load the data again with the correct sub-folder
train_ds = tf.keras.utils.image_dataset_from_directory(
    new_path2,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    new_path2,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Folders found: ['kidney US images']
Found 9416 files belonging to 1 classes.
Using 7533 files for training.
Found 9416 files belonging to 1 classes.
Using 1883 files for validation.


In [6]:
# Go one more level deeper into the 'kidney US images' folder
final_path = os.path.join(new_path2, 'kidney US images')

# Verification: This should finally show the class folders (e.g., 'Stone' and 'Normal')
print("Actual Class Folders:", os.listdir(final_path))

# Load the data one last time
train_ds = tf.keras.utils.image_dataset_from_directory(
    final_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    final_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Actual Class Folders: ['my dataset final 512x512(implemented)']
Found 9416 files belonging to 1 classes.
Using 7533 files for training.
Found 9416 files belonging to 1 classes.
Using 1883 files for validation.


In [11]:
# Go to the final level where the 'Stone' and 'Normal' folders live
ultimate_path = os.path.join(final_path, 'my dataset final 512x512(implemented)')

# Verification: This MUST show the labels now
print("The Labels are:", os.listdir(ultimate_path))

# Load the data
train_ds = tf.keras.utils.image_dataset_from_directory(
    ultimate_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    ultimate_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

The Labels are: ['Normal', 'stone']
Found 9416 files belonging to 2 classes.
Using 7533 files for training.
Found 9416 files belonging to 2 classes.
Using 1883 files for validation.


In [12]:
# Create the base model from the pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False  # Freeze the base to keep existing knowledge

# Add your custom layers on top for Kidney Stones
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation='sigmoid') # Output: 0 for Normal, 1 for Stone
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [13]:
# 1. Clear memory just in case
tf.keras.backend.clear_session()

# 2. Reload data with a smaller Batch Size (16 instead of 32)
# This uses less RAM and helps the progress bar start faster
train_ds = tf.keras.utils.image_dataset_from_directory(
    ultimate_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=16  # Smaller batch
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    ultimate_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=16
)

# 3. Add a "Prefetch" - this prepares the next images while the AI is learning
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Start Training again
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=5)

Found 9416 files belonging to 2 classes.
Using 7533 files for training.
Found 9416 files belonging to 2 classes.
Using 1883 files for validation.
Epoch 1/5
471/471 ━━━━━━━━━━━━━━━━━━━━ 64s 94ms/step - accuracy: 0.9652 - loss: 0.1154 - val_accuracy: 0.9793 - val_loss: 0.0594
Epoch 2/5
471/471 ━━━━━━━━━━━━━━━━━━━━ 17s 36ms/step - accuracy: 0.9923 - loss: 0.0328 - val_accuracy: 0.9931 - val_loss: 0.0326
Epoch 3/5
471/471 ━━━━━━━━━━━━━━━━━━━━ 17s 37ms/step - accuracy: 0.9952 - loss: 0.0205 - val_accuracy: 0.9947 - val_loss: 0.0235
Epoch 4/5
471/471 ━━━━━━━━━━━━━━━━━━━━ 18s 38ms/step - accuracy: 0.9975 - loss: 0.0144 - val_accuracy: 0.9952 - val_loss: 0.0182
Epoch 5/5
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9983 - loss: 0.0109 - val_accuracy: 0.9973 - val_loss: 0.0148


In [14]:
model.save('kidney_model.h5')
print("Model saved as kidney_model.h5")

Model saved as kidney_model.h5


In [15]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

# Upload an image
uploaded = files.upload()

for fn in uploaded.keys():
  # Prepare the image
  img = image.load_img(fn, target_size=(224, 224))
  x = image.img_to_array(img)
  x = np.expand_dims(x, axis=0)
  x = x / 255.0  # Normalize like we did in training

  # Predict
  classes = model.predict(x)

  if classes[0] > 0.5:
    print(f"\nRESULT: {fn} is likely a KIDNEY STONE")
  else:
    print(f"\nRESULT: {fn} is NORMAL")

Saving WhatsApp Image 2026-03-12 at 10.24.27 AM.jpeg to WhatsApp Image 2026-03-12 at 10.24.27 AM.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step

RESULT: WhatsApp Image 2026-03-12 at 10.24.27 AM.jpeg is NORMAL


In [16]:
# Check if there is a 'labels' or 'annotations' folder
print("Inside the new archive:", os.listdir(path1))

Inside the new archive: ['Normal', 'stone']


In [17]:
# The two datasets we downloaded earlier
paths = [path1, ultimate_path]

print("Combining data from:")
for p in paths:
    print(f"- {p}")

Combining data from:
- /root/.cache/kagglehub/datasets/imtkaggleteam/kidney-stone-classification-and-object-detection/versions/1
- /root/.cache/kagglehub/datasets/makiw4/kidney-dataset-stone-and-normal/versions/1/Kidney Ultrasound Images Stone and No Stone/kidney US images/my dataset final 512x512(implemented)


In [18]:
# Standard settings for your project
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Loading the first set
train_ds1 = tf.keras.utils.image_dataset_from_directory(
    paths[0], validation_split=0.2, subset="training", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

# Loading the second set
train_ds2 = tf.keras.utils.image_dataset_from_directory(
    paths[1], validation_split=0.2, subset="training", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

# Combine them using the .concatenate() method
full_train_ds = train_ds1.concatenate(train_ds2).prefetch(buffer_size=tf.data.AUTOTUNE)

Found 9416 files belonging to 2 classes.
Using 7533 files for training.
Found 9416 files belonging to 2 classes.
Using 7533 files for training.


In [19]:
# Re-running the fit command on the combined data
history_combined = model.fit(
    full_train_ds,
    epochs=5
)

Epoch 1/5
942/942 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 0.9990 - loss: 0.0077
Epoch 2/5
942/942 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 0.9996 - loss: 0.0050
Epoch 3/5
942/942 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 0.9999 - loss: 0.0034
Epoch 4/5
942/942 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 1.0000 - loss: 0.0024
Epoch 5/5
942/942 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 1.0000 - loss: 0.0017


In [20]:
from google.colab import files
model.save('kidney_model_final.h5')
files.download('kidney_model_final.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# Re-building the top layers for 3 categories
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation='softmax') # 3 nodes for 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Required for more than 2 classes
    metrics=['accuracy']
)

In [22]:
import kagglehub
import os

# Download both fetus datasets
path_fetus1 = kagglehub.dataset_download("orvile/ultrasound-fetus-dataset")
path_fetus2 = kagglehub.dataset_download("orvile/dataset-for-fetus-framework")

print("Fetus Data 1:", os.listdir(path_fetus1))
print("Fetus Data 2:", os.listdir(path_fetus2))

100%|██████████| 604M/604M [00:04<00:00, 141MB/s]

Extracting files...


Using Colab cache for faster access to the 'dataset-for-fetus-framework' dataset.
Fetus Data 1: ['Ultrasound Fetus Dataset', 'ultrasound_fetus.csv']
Fetus Data 2: ['Dataset for Fetus Framework', 'ObjectDetection.csv']


In [23]:
import tensorflow as tf

# Settings for the Fetus Brain
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Load the fetus data
# Note: Ensure the path points to the folder containing subfolders like '1st_trimester', '2nd', etc.
fetus_train_ds = tf.keras.utils.image_dataset_from_directory(
    path_fetus1,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Build a Fetus-Specific Model
base_model_fetus = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model_fetus.trainable = False

fetus_model = tf.keras.Sequential([
    base_model_fetus,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax') # Assuming 4 age categories/trimesters
])

fetus_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the Fetus model
fetus_model.fit(fetus_train_ds, epochs=10)
fetus_model.save('fetus_model.h5')

Found 6033 files belonging to 1 classes.
Using 4827 files for training.
Epoch 1/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 30s 69ms/step - accuracy: 0.9967 - loss: 0.0087
Epoch 2/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 3/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 19s 56ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 4/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 21s 57ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 5/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 6/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 17s 57ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 7/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 8/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 17s 57ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 9/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 21s 59ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Epoch 10/10
302/302 ━━━━━━━━━━━━━━━━━━━━ 18s 58ms/step - accuracy: 1.0000 - loss: 0.0000e+00


In [26]:
# 1. Load the models into memory
# Make sure these filenames match what you saved earlier
kidney_model = tf.keras.models.load_model('kidney_model_final.h5')
fetus_model = tf.keras.models.load_model('fetus_model.h5')

# 2. Re-run the test
print("--- KIDNEY SECTION TEST ---")
try:
    print(analyze_ultrasound(sample_stone_path, section="Kidney"))
except Exception as e:
    print(f"Error in Kidney Test: {e}")

print("\n--- FETUS SECTION TEST ---")
try:
    print(analyze_ultrasound(sample_fetus_path, section="Fetus"))
except Exception as e:
    print(f"Error in Fetus Test: {e}")

--- KIDNEY SECTION TEST ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
STONE DETECTED. Estimated Size: 14.50 mm

--- FETUS SECTION TEST ---
Error in Fetus Test: [Errno 21] Is a directory: '/root/.cache/kagglehub/datasets/orvile/ultrasound-fetus-dataset/versions/2/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset'


In [31]:
# Updated logic to find a real image file inside the nested folders
fetus_root = "/root/.cache/kagglehub/datasets/orvile/ultrasound-fetus-dataset/versions/2/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset"
first_subfolder = os.path.join(fetus_root, os.listdir(fetus_root)[0])
sample_fetus_path = os.path.join(first_subfolder, os.listdir(first_subfolder)[0])

print("\n--- FETUS SECTION TEST ---")
try:
    print(analyze_ultrasound(sample_fetus_path, section="Fetus"))
except Exception as e:
    print(f"Error in Fetus Test: {e}")


--- FETUS SECTION TEST ---
Error in Fetus Test: [Errno 21] Is a directory: '/root/.cache/kagglehub/datasets/orvile/ultrasound-fetus-dataset/versions/2/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data'


In [33]:
from google.colab import files
files.download('kidney_model_final.h5')
files.download('fetus_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>